In [1]:
import pandas as pd

# Load the news dataset
df = pd.read_csv("../data/finbert/news_finbert_ready_20250804_160853.csv")

# Preview the first few rows
print(df.head())

  stock_symbol   published_date  \
0         AAPL  20250804T123000   
1         AAPL  20250804T102002   
2         AAPL  20250804T090901   
3         AAPL  20250804T074139   
4         AAPL  20250803T220500   

                                         title_clean  \
0  APPLE INC. SHAREHOLDER ALERT Bernstein Liebhar...   
1  Is Fidelity High Dividend ETF a Strong ETF Rig...   
2  Elon Musk Slams Mark Zuckerberg's Hiring Spree...   
3  Google Loses Epic Games Appeal As Court Orders...   
4     Should You Buy Sirius XM Stock After Earnings?   

                                        finbert_text  \
0  APPLE INC. SHAREHOLDER ALERT Bernstein Liebhar...   
1  Is Fidelity High Dividend ETF a Strong ETF Rig...   
2  Elon Musk Slams Mark Zuckerberg's Hiring Spree...   
3  Google Loses Epic Games Appeal As Court Orders...   
4  Should You Buy Sirius XM Stock After Earnings?...   

                                         article_url    news_source  \
0  https://www.benzinga.com/pressreleases/25

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

# Load FinBERT pretrained for financial sentiment
finbert = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")

# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer)


Device set to use mps:0


In [3]:
# Example on a few rows to test
sample_texts = df['title_clean'].head(10).tolist()

# Run FinBERT on these
results = sentiment_analyzer(sample_texts)

# Add results back to DataFrame
df_sample = df.head(10).copy()
df_sample['sentiment'] = [r['label'] for r in results]
df_sample['confidence'] = [r['score'] for r in results]

print(df_sample[['title_clean', 'sentiment', 'confidence']])


                                         title_clean sentiment  confidence
0  APPLE INC. SHAREHOLDER ALERT Bernstein Liebhar...   Neutral    0.999968
1  Is Fidelity High Dividend ETF a Strong ETF Rig...  Positive    0.997437
2  Elon Musk Slams Mark Zuckerberg's Hiring Spree...  Positive    0.970238
3  Google Loses Epic Games Appeal As Court Orders...  Negative    0.999990
4     Should You Buy Sirius XM Stock After Earnings?   Neutral    0.998449
5  Apple's New 'Answers' Team Developing ChatGPT ...   Neutral    0.997879
6  Undervalued and Profitable: 3 Artificial Intel...  Positive    0.998830
7  Here's How Much You Would Have Made Owning App...   Neutral    0.999969
8  Apple Q3 Earnings Beat Estimates, Services Dri...  Positive    1.000000
9  Tech Stocks Trace A Negative Pattern; Trump We...  Negative    0.999999


In [ ]:
# Warning: this can be slow for large data
df['sentiment'] = df['title_clean'].apply(lambda x: sentiment_analyzer(x)[0]['label'])
df['confidence'] = df['title_clean'].apply(lambda x: sentiment_analyzer(x)[0]['score'])

# Save to CSV
df.to_csv("../data/finbert/news_with_sentiment.csv", index=False)
